In [ ]:
from datasets import load_dataset

In [2]:
ds = load_dataset("pszemraj/multi_fc")

In [ ]:
ds

### Remove 'weird' labels 0, 1, 10, 2, 3, 4

In [4]:
labels_to_remove = {"0", "1", "10", "2", "3", "4"}
ds_train_filtered = ds["train"].filter(lambda x: x["label"] not in labels_to_remove)
ds_validation_filtered = ds["validation"].filter(lambda x: x["label"] not in labels_to_remove)

## Label Mapping to FEVER labels

In [5]:
label_mapping = {

    '2 pinnochios': 'REFUTES',
    '3 pinnochios': 'REFUTES',
    '4 pinnochios': 'REFUTES',

    None: 'NOT ENOUGH INFO',

    # negative or debunking labels
    'a little baloney': 'REFUTES',
    'a lot of baloney': 'REFUTES',
    'bogus warning': 'REFUTES',
    'cherry picks': 'REFUTES',

    # ambiguous or commentary
    'commentary!': 'NOT ENOUGH INFO',
    'compromise': 'NOT ENOUGH INFO',

    # conclusion based
    'conclusion: accurate': 'SUPPORTS',
    'conclusion: false': 'REFUTES',
    'conclusion: unclear': 'NOT ENOUGH INFO',

    # authorship and correctness signals
    'authorship confirmed!': 'SUPPORTS',
    'confirmed authorship!': 'SUPPORTS',
    'correct': 'SUPPORTS',
    'correct attribution': 'SUPPORTS',
    'correct attribution!': 'SUPPORTS',

    # determinations
    'determination: a stretch': 'REFUTES',
    'determination: barely true': 'REFUTES',
    'determination: false': 'REFUTES',
    'determination: huckster propaganda': 'REFUTES',
    'determination: misleading': 'REFUTES',
    'determination: mostly true': 'SUPPORTS',
    'determination: true': 'SUPPORTS',

    # disputed or exaggerated
    'disputed!': 'REFUTES',
    'distorts the facts': 'REFUTES',
    'exaggerated': 'REFUTES',
    'exaggerates': 'REFUTES',
    'facebook scams': 'REFUTES',

    # fact-based
    'fact': 'SUPPORTS',
    'factscan score: false': 'REFUTES',
    'factscan score: misleading': 'REFUTES',
    'factscan score: true': 'SUPPORTS',

    # clearly false or fictional claims
    'fake': 'REFUTES',
    'fake news': 'REFUTES',
    'false': 'REFUTES',
    'fiction': 'REFUTES',
    'fiction!': 'REFUTES',
    'fiction! & satire!': 'REFUTES',
    'full flop': 'REFUTES',

    # ambiguous cases
    'grass roots movement!': 'NOT ENOUGH INFO',
    'half flip': 'REFUTES',
    'half true': 'REFUTES',
    'half-true': 'REFUTES',

    # in the works
    'in the works': 'NOT ENOUGH INFO',
    'in-between': 'NOT ENOUGH INFO',
    'in-the-green': 'SUPPORTS',
    'in-the-red': 'REFUTES',
    'inaccurate attribution!': 'REFUTES',
    'incorrect': 'REFUTES',
    'incorrect attribution!': 'REFUTES',
    'investigation pending!': 'NOT ENOUGH INFO',
    'legend': 'NOT ENOUGH INFO',

    # misattributed, miscaptioned, misleading
    'misattributed': 'REFUTES',
    'miscaptioned': 'REFUTES',
    'misleading': 'REFUTES',
    'misleading recommendations': 'REFUTES',
    'misleading!': 'REFUTES',

    # mostly false, mostly fiction, mostly true, mostly truth, mostly correct
    'mixture': 'NOT ENOUGH INFO',
    'mostly false': 'REFUTES',
    'mostly fiction!': 'REFUTES',
    'mostly true': 'SUPPORTS',
    'mostly truth!': 'SUPPORTS',
    'mostly-correct': 'SUPPORTS',
    'mostly_false': 'REFUTES',
    'mostly_true': 'SUPPORTS',

    # needs context, no evidence, no flip, none, not the whole story, not yet rated
    'needs context': 'NOT ENOUGH INFO',
    'no evidence': 'NOT ENOUGH INFO',
    'no flip': 'NOT ENOUGH INFO',
    'none': 'NOT ENOUGH INFO',
    'not the whole story': 'NOT ENOUGH INFO',
    'not yet rated': 'NOT ENOUGH INFO',

    'opinion!': 'NOT ENOUGH INFO',

    # outdated, pants on fire, partially true, partly true
    'outdated': 'REFUTES',
    'outdated!': 'REFUTES',

    'pants on fire!': 'REFUTES',
    'partially true': 'REFUTES',
    'partly true': 'REFUTES',

    'previously truth! now resolved!': 'SUPPORTS',
    'promise broken': 'REFUTES',
    'promise kept': 'SUPPORTS',

    'rating: false': 'REFUTES',
    'scam': 'REFUTES',
    'scam!': 'REFUTES',

    'some baloney': 'REFUTES',
    'spins the facts': 'REFUTES',
    'stalled': 'NOT ENOUGH INFO',
    'statirical reports': 'REFUTES',

    # different degrees of true labels 
    'true': 'SUPPORTS',
    'true messages': 'SUPPORTS',
    'truth!': 'SUPPORTS',
    'truth! & disputed!': 'NOT ENOUGH INFO',
    'truth! & fiction!': 'REFUTES',
    'truth! & misleading!': 'REFUTES',
    'truth! & outdated!': 'REFUTES',
    'truth! & unproven!': 'NOT ENOUGH INFO',

    # unobservable, unproven, unsubstantiated messages, unsupported, unverified
    'understated': 'SUPPORTS',

    'unobservable': 'NOT ENOUGH INFO',
    'unproven': 'NOT ENOUGH INFO',
    'unproven!': 'NOT ENOUGH INFO',
    'unsubstantiated messages': 'NOT ENOUGH INFO',
    'unsupported': 'NOT ENOUGH INFO',
    'unverified': 'NOT ENOUGH INFO',

    'verdict: false': 'REFUTES',
    'verdict: true': 'SUPPORTS',
    'verdict: unsubstantiated': 'NOT ENOUGH INFO',

    'verified': 'SUPPORTS',

    'virus!': 'REFUTES',

    'we rate this claim false': 'REFUTES'
}

def apply_label_mapping(entry):
    raw_label = entry["label"]
    entry["label"] = label_mapping.get(raw_label, "NOT ENOUGH INFO")
    return entry

In [6]:
multifc_train_mapped = ds_train_filtered.map(apply_label_mapping)
multifc_validation_mapped = ds_validation_filtered.map(apply_label_mapping)

In [ ]:
print("Train sample:")
print(multifc_train_mapped[1])
print("\nValidation sample:")
print(multifc_validation_mapped[0])

### No test labels available - split train into train + test

In [8]:
split_dataset = multifc_train_mapped.train_test_split(test_size=0.2, seed=42)

In [9]:
multi_fc_train = split_dataset['train']
multi_fc_test = split_dataset['test']
multi_fc_validation = multifc_validation_mapped

In [ ]:
print(multi_fc_train.shape[0])
print(multi_fc_test.shape[0])
print(multi_fc_validation.shape[0])

In [ ]:
multi_fc_train

In [ ]:
multi_fc_train[0]

## Reshape datasets

In [13]:
from collections import OrderedDict

In [14]:
def reshape_entry(entry, index):
    new_entry = OrderedDict()
    new_entry["unique_id"] = index                     # Sequential id starting from 0
    new_entry["claim"] = entry["claim"]
    new_entry["label"] = entry["label"]
    new_entry["claimURL"] = entry["claimURL"]
    new_entry["human_verified"] = "NO"
    new_entry["verifiable"] = "NOT VERIFIABLE" if entry["label"] == "NOT ENOUGH INFO" else "VERIFIABLE"
    
    return new_entry

In [15]:
columns_to_remove = [
    "claimID", "reason", "categories", "speaker",
    "checker", "tags", "article title", "publish date",
    "climate", "entities"
]

In [ ]:
multi_fc_train_final = multi_fc_train.map(reshape_entry, with_indices=True, remove_columns=columns_to_remove)
multi_fc_test_final = multi_fc_test.map(reshape_entry, with_indices=True, remove_columns=columns_to_remove)
multi_fc_validation_final = multi_fc_validation.map(reshape_entry, with_indices=True, remove_columns=columns_to_remove)

In [ ]:
print(multi_fc_train_final)

In [ ]:
df_train_final = multi_fc_train_final.to_pandas()
df_test_final = multi_fc_test_final.to_pandas()
df_validation_final = multi_fc_validation_final.to_pandas()

desired_order = ["unique_id", "claim", "label", "claimURL", "human_verified", "verifiable"]

df_train_final = df_train_final[desired_order]
df_test_final = df_test_final[desired_order]
df_validation_final = df_validation_final[desired_order]

print("Train DataFrame:")
print(df_train_final.head())

print("\nTest DataFrame:")
print(df_test_final.head())

print("\nValidation DataFrame:")
print(df_validation_final.head())


In [20]:
import os

In [21]:
df_train_final.to_json(os.path.join(os.getcwd(), "multifc", "train.json"), orient="records", lines=True)

In [22]:
df_test_final.to_json(os.path.join(os.getcwd(), "multifc", "test.json"), orient="records", lines=True)

In [23]:
df_validation_final.to_json(os.path.join(os.getcwd(), "multifc", "validation.json"), orient="records", lines=True)